# Lyα $P_{1D}$: porównanie fitu linear i one-loop

Notebook wykonuje pełne porównanie modeli `linear` i `one_loop` dla tego samego pliku teorii i tych samych danych. Pokazuje:

- wynik wielu seedów optymalizatora;
- $\chi^2_{\rm linear}$, $\chi^2_{\rm 1-loop}$ i $\Delta\chi^2$;
- parametry best fit i ich odległość od granic;
- dane oraz oba modele dla każdego redshiftu;
- pullsy i wkład poszczególnych redshiftów do $\Delta\chi^2$;
- surowe kanały $P_{\delta\delta}$, $P_{\delta\theta}$ i $P_{\theta\theta}$;
- opcjonalny skan cutoffu $k_{\rm UV}$.

Notebook nie wystawia automatycznego werdyktu. Tabele zawierają wartości odniesienia z publikacji, ale ocena jakości wyniku pozostaje po stronie użytkownika.


## 1. Instalacja

Z głównego katalogu projektu zalecane jest jednorazowe wykonanie:

```bash
python -m pip install -e .
python -m pip install pandas matplotlib jupyter
```

Do fitowania zapisanego `.npz` nie jest potrzebny CLASS.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, "to_string") else value)

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8-whitegrid")

candidates = [
    Path.cwd(),
    Path.cwd() / "lyalpha_one_loop",
    Path.cwd().parent,
    Path.cwd().parent / "lyalpha_one_loop",
]
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Nie znaleziono głównego katalogu projektu.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lyalpha_pt.data import load_dr12
from lyalpha_pt.fit import EffectiveModelFit, PARAMETER_NAMES, write_fit_result
from lyalpha_pt.theory import load_theory

print("PROJECT_ROOT =", PROJECT_ROOT)
print("Python =", sys.version.split()[0])


## 2. Konfiguracja

Obecnie wskazany jest dostępny plik `smoke`. Po obliczeniu właściwej teorii na klastrze zmień tylko `THEORY_FILENAME`, na przykład na `lcdm_2021_massless_precision.npz`.

`RUN_NEW_FITS=True` wykonuje oba fity od początku. Po pierwszym uruchomieniu można ustawić `False`, aby wczytać wynik zapisany przez notebook.


In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
THEORY_FILENAME = "lcdm_2021_massless_smoke.npz"
THEORY_PATH = PROJECT_ROOT / "results" / THEORY_FILENAME
OUTPUT_DIR = PROJECT_ROOT / "results" / f"linear_vs_one_loop_{THEORY_PATH.stem}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVED_COMPARISON_PATH = OUTPUT_DIR / "fit_comparison.json"

K_UV_CUT = 20.0  # h/Mpc
COVARIANCE_MODE = "paper_diag"
RUN_NEW_FITS = True
FIT_SEEDS = [12345, 23456, 34567]
DE_MAXITER = 300
DE_POPSIZE = 20
SAVE_FIGURES = True

# Skan jest kosztowniejszy, dlatego domyślnie pozostaje wyłączony.
RUN_CUTOFF_SCAN = False
CUTOFFS = [10.0, 15.0, 20.0]
CUTOFF_SCAN_SEEDS = [12345, 23456, 34567]

BENCHMARK_TARGETS = {
    "lcdm_2021_massless": {
        "linear": 206.5,
        "one_loop": 193.4,
        "delta": -13.1,
        "note": "arXiv:2011.03050, massless-neutrino LCDM",
    },
    "lcdm_2022_planck": {
        "linear": 206.29,  # wartość przybliżona: 192.89 + 13.4
        "one_loop": 192.89,
        "delta": -13.4,
        "note": "arXiv:2210.06117, Planck-2018 LCDM",
    },
}

print("Theory:", THEORY_PATH)
print("Output:", OUTPUT_DIR)


## 3. Dane, teoria i profil numeryczny

Ta sekcja pokazuje faktyczną konfigurację zapisaną wewnątrz pliku. Ma to szczególne znaczenie dla `smoke`, ponieważ rzadka siatka pętli może silnie zniekształcić projekcję 3D→1D.


In [ ]:
dataset = load_dr12(DATA_DIR)
theory = load_theory(THEORY_PATH)
model_name = theory.metadata["model"]["name"]
numerics = theory.metadata.get("numerics", {})
targets = BENCHMARK_TARGETS.get(
    model_name,
    {"linear": np.nan, "one_loop": np.nan, "delta": np.nan, "note": "no target configured"},
)

n_loop = len(theory.k_loop)
if n_loop <= 30:
    quality_hint = "smoke / software test"
elif n_loop >= 150:
    quality_hint = "precision-like"
else:
    quality_hint = "production-like"

configuration_table = pd.DataFrame({
    "quantity": [
        "model", "benchmark", "data points", "redshift bins",
        "linear grid points", "loop grid points", "loop k range [h/Mpc]",
        "k_trust [h/Mpc]", "projection k_uv [h/Mpc]", "quality hint",
        "loop source", "loop weight", "bundle digest",
    ],
    "value": [
        model_name, targets["note"], dataset.n_data, len(dataset.z_unique),
        len(theory.k_input), n_loop, f"{theory.k_loop[0]:.4g}--{theory.k_loop[-1]:.4g}",
        numerics.get("k_trust", np.nan), K_UV_CUT, quality_hint,
        theory.metadata.get("loop_source"), theory.loop_weight,
        theory.metadata["bundle_digest"],
    ],
})
display(configuration_table)

if n_loop <= 30:
    print(
        "UWAGA: to jest rzadka siatka smoke. Wynik one-loop poniżej służy do "
        "sprawdzenia notebooka i nie powinien być porównywany naukowo z publikacją."
    )


## 4. Surowe kanały one-loop na kontrolowanym zakresie $k\leq2\,h/\mathrm{Mpc}$

Najpierw oglądamy teorię przed projekcją i przed dopasowaniem nuisance parameters. Rysujemy stosunek każdego kanału one-loop do tree level.


In [ ]:
CHANNEL_LABELS = {
    "dd": r"$P_{\delta\delta}/P_{\rm tree}$",
    "dtheta": r"$P_{\delta\theta}/P_{\rm tree}$",
    "thetatheta": r"$P_{\theta\theta}/P_{\rm tree}$",
}
channel_names = theory.metadata.get("channels", ["dd", "dtheta", "thetatheta"])
selected_redshifts = [3.0, 3.6, 4.2]
k_control_max = float(numerics.get("k_trust", 2.0))

fig, axes = plt.subplots(1, len(selected_redshifts), figsize=(15, 4.5), sharey=True)
for ax, z in zip(axes, selected_redshifts):
    iz = int(np.argmin(np.abs(theory.z - z)))
    mask = theory.k_loop <= k_control_max
    k = theory.k_loop[mask]
    tree = theory.p_tree[iz, mask]
    for ichannel, channel in enumerate(channel_names):
        ratio = theory.channels_one_loop[iz, mask, ichannel] / tree
        ax.plot(k, ratio, lw=2, label=CHANNEL_LABELS.get(channel, channel))
    ax.axhline(1, color="black", lw=1)
    ax.set_xscale("log")
    ax.set_xlabel(r"$k\,[h/\mathrm{Mpc}]$")
    ax.set_title(rf"$z={theory.z[iz]:.1f}$")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("one-loop / tree")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.015),
    ncol=3, frameon=False,
)
fig.suptitle(rf"Raw SPT channels for $k\leq{k_control_max:g}\,h/\mathrm{{Mpc}}$", y=1.10)
fig.tight_layout(rect=[0, 0, 1, 0.88])
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR / "raw_one_loop_channel_ratios.png", dpi=180, bbox_inches="tight")
plt.show()


## 5. Rozdzielenie $P_{22}$ i $P_{13}$

Poniższy wykres pokazuje kanał gęstościowy. Jest to diagnostyka surowej teorii, niezależna od projekcji $P_{1D}$ i fitu.


In [ ]:
fig, axes = plt.subplots(1, len(selected_redshifts), figsize=(15, 4.5), sharey=True)
for ax, z in zip(axes, selected_redshifts):
    iz = int(np.argmin(np.abs(theory.z - z)))
    mask = theory.k_loop <= k_control_max
    k = theory.k_loop[mask]
    tree = theory.p_tree[iz, mask]
    ax.plot(k, theory.loop_weight * theory.p22[iz, mask, 0] / tree, lw=2, label=r"$P_{22}^{\delta\delta}/P_{\rm tree}$")
    ax.plot(k, theory.loop_weight * theory.p13[iz, mask, 0] / tree, lw=2, label=r"$P_{13}^{\delta\delta}/P_{\rm tree}$")
    ax.plot(
        k, theory.loop_weight * (theory.p22[iz, mask, 0] + theory.p13[iz, mask, 0]) / tree,
        lw=2, ls="--", label=r"$(P_{22}+P_{13})/P_{\rm tree}$",
    )
    ax.axhline(0, color="black", lw=1)
    ax.set_xscale("log")
    ax.set_xlabel(r"$k\,[h/\mathrm{Mpc}]$")
    ax.set_title(rf"$z={theory.z[iz]:.1f}$")
axes[0].set_ylabel("loop correction / tree")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.92])
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR / "raw_p22_p13_density.png", dpi=180, bbox_inches="tight")
plt.show()


## 6. Wielostartowy fit linear i one-loop

Oba modele są dopasowywane niezależnie w tym samym sześciowymiarowym boxie numerycznym. Dzięki temu $\Delta\chi^2$ nie zależy od wcześniejszego zapisanego minimum.


In [ ]:
fitters = {
    mode: EffectiveModelFit(
        dataset, theory, mode=mode, k_uv_cut=K_UV_CUT, covariance_mode=COVARIANCE_MODE
    )
    for mode in ("linear", "one_loop")
}

best_theta = {}
best_results = {}
multistart_rows = []

if RUN_NEW_FITS:
    for mode, fitter in fitters.items():
        runs = []
        print("\n" + "=" * 72)
        print("Fitting", mode)
        for seed in FIT_SEEDS:
            start = time.perf_counter()
            result = fitter.fit_once(
                seed=seed, bounds=fitter.bounds_broad(),
                de_maxiter=DE_MAXITER, de_popsize=DE_POPSIZE,
            )
            elapsed = time.perf_counter() - start
            chi2_value = fitter.chi2(result.x)
            runs.append(result)
            multistart_rows.append({
                "mode": mode, "seed": seed, "chi2": chi2_value,
                "success": bool(result.success), "elapsed_s": elapsed,
            })
            print(f"seed={seed}: chi2={chi2_value:.9f}, time={elapsed:.1f} s")
        best = min(runs, key=lambda result: fitter.chi2(result.x))
        best_results[mode] = best
        best_theta[mode] = np.asarray(best.x, dtype=float)
else:
    if not SAVED_COMPARISON_PATH.is_file():
        raise FileNotFoundError("Brak zapisanego fit_comparison.json; ustaw RUN_NEW_FITS=True.")
    with SAVED_COMPARISON_PATH.open(encoding="utf-8") as handle:
        saved_comparison = json.load(handle)
    for mode in ("linear", "one_loop"):
        record = saved_comparison["fits"][mode]
        best_theta[mode] = np.array([record["parameters"][name] for name in PARAMETER_NAMES])
    multistart_rows = saved_comparison.get("multistart", [])

multistart_table = pd.DataFrame(multistart_rows)
display(multistart_table)


## 7. Główne wyniki i porównanie z benchmarkiem

Kolumny `target` są informacyjne. Dla `lcdm_2022_planck` liniowe 206.29 jest wartością przybliżoną otrzymaną z $192.89+13.4$, a nie osobno zacytowanym dokładnym minimum.


In [ ]:
summaries = {mode: fitters[mode].summary(best_theta[mode]) for mode in fitters}
chi2_linear = summaries["linear"]["chi2"]
chi2_loop = summaries["one_loop"]["chi2"]
delta_chi2 = chi2_loop - chi2_linear

result_table = pd.DataFrame([
    {
        "quantity": "chi2 linear", "result": chi2_linear,
        "target": targets["linear"], "result_minus_target": chi2_linear - targets["linear"],
    },
    {
        "quantity": "chi2 one-loop", "result": chi2_loop,
        "target": targets["one_loop"], "result_minus_target": chi2_loop - targets["one_loop"],
    },
    {
        "quantity": "Delta chi2 (one-loop - linear)", "result": delta_chi2,
        "target": targets["delta"], "result_minus_target": delta_chi2 - targets["delta"],
    },
])
display(result_table)

summary_table = pd.DataFrame([
    {
        "mode": mode, "chi2": summary["chi2"], "dof": summary["dof"],
        "chi2_per_dof": summary["chi2_per_dof"], "p_value": summary["p_value"],
    }
    for mode, summary in summaries.items()
])
display(summary_table)


## 8. Parametry best fit i granice

Tabela nie oznacza parametrów jako dobre lub złe. Pokazuje surowe wartości, przesunięcie linear→one-loop oraz znormalizowaną odległość od najbliższej granicy.


In [ ]:
parameter_table = pd.DataFrame({
    "parameter": PARAMETER_NAMES,
    "linear": best_theta["linear"],
    "one_loop": best_theta["one_loop"],
})
parameter_table["one_loop_minus_linear"] = parameter_table["one_loop"] - parameter_table["linear"]
display(parameter_table)

boundary_rows = []
for mode, fitter in fitters.items():
    for row in fitter.boundary_diagnostics(best_theta[mode], fitter.bounds_broad()):
        boundary_rows.append({"mode": mode, **row})
boundary_table = pd.DataFrame(boundary_rows)
display(boundary_table)


## 9. $\chi^2$ według redshiftu

Najważniejsza kolumna to `delta_chi2`: wartości ujemne wskazują redshifty, w których one-loop poprawia dopasowanie względem linear.


In [ ]:
chi2_linear_z = pd.DataFrame(fitters["linear"].chi2_by_redshift(best_theta["linear"]))
chi2_loop_z = pd.DataFrame(fitters["one_loop"].chi2_by_redshift(best_theta["one_loop"]))
chi2_by_z = chi2_linear_z.rename(columns={"chi2": "chi2_linear"}).merge(
    chi2_loop_z[["z", "chi2"]].rename(columns={"chi2": "chi2_one_loop"}), on="z"
)
chi2_by_z["delta_chi2"] = chi2_by_z["chi2_one_loop"] - chi2_by_z["chi2_linear"]
display(chi2_by_z)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
colors = np.where(chi2_by_z["delta_chi2"] < 0, "tab:green", "tab:red")
ax.bar(chi2_by_z["z"].astype(str), chi2_by_z["delta_chi2"], color=colors)
ax.axhline(0, color="black", lw=1)
ax.set_xlabel("z")
ax.set_ylabel(r"$\Delta\chi^2_z=\chi^2_{\rm 1-loop}-\chi^2_{\rm linear}$")
ax.set_title(rf"Contributions to total $\Delta\chi^2={delta_chi2:.3f}$")
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR / "delta_chi2_by_redshift.png", dpi=180, bbox_inches="tight")
plt.show()


## 10. Dane oraz best fit linear i one-loop

Rysujemy $k_vP_{1D}(k_v)/\pi$. Obie krzywe są rzeczywistymi predykcjami swoich niezależnych best fitów.


In [ ]:
models = {mode: fitters[mode].model(best_theta[mode]) for mode in fitters}
pulls = {mode: (models[mode] - dataset.p1d) / dataset.sigma_total for mode in fitters}

def plot_quantity(kv, p1d):
    return kv * p1d / np.pi

z_values = dataset.z_unique
ncols = 2
nrows = int(np.ceil(len(z_values) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.0 * nrows))
axes = np.asarray(axes).ravel()

for ax, z in zip(axes, z_values):
    mask = np.isclose(dataset.z, z)
    order = np.argsort(dataset.k_velocity[mask])
    kv = dataset.k_velocity[mask][order]
    row = chi2_by_z.loc[np.isclose(chi2_by_z["z"], z)].iloc[0]
    ax.errorbar(
        kv, plot_quantity(kv, dataset.p1d[mask][order]),
        yerr=plot_quantity(kv, dataset.sigma_total[mask][order]),
        fmt="o", ms=3.7, capsize=2, color="black", ecolor="0.55", label="BOSS data",
    )
    ax.plot(kv, plot_quantity(kv, models["linear"][mask][order]), lw=2, label="linear")
    ax.plot(kv, plot_quantity(kv, models["one_loop"][mask][order]), lw=2, ls="--", label="one-loop")
    ax.set_xscale("log")
    ax.set_xlabel(r"$k_v\,[\mathrm{s/km}]$")
    ax.set_ylabel(r"$k_vP_{1D}(k_v)/\pi$")
    ax.set_title(
        rf"$z={z:.1f}$: $\chi^2_{{\rm lin}}={row['chi2_linear']:.2f}$, "
        rf"$\chi^2_{{\rm 1L}}={row['chi2_one_loop']:.2f}$"
    )
    ax.grid(alpha=0.25)
for ax in axes[len(z_values):]:
    ax.set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.995), ncol=3, frameon=False)
fig.suptitle(
    rf"$\chi^2_{{\rm linear}}={chi2_linear:.3f}$, "
    rf"$\chi^2_{{\rm 1-loop}}={chi2_loop:.3f}$, $\Delta\chi^2={delta_chi2:.3f}$",
    y=1.015,
)
fig.tight_layout(rect=[0, 0, 1, 0.955])
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR / "linear_vs_one_loop_panels.png", dpi=180, bbox_inches="tight")
plt.show()


## 11. Pullsy linear i one-loop

To porównanie pokazuje, czy poprawa lub pogorszenie $\chi^2$ ma postać systematycznej zmiany z $k_v$, czy wynika z kilku punktów.


In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.6 * nrows), sharey=True)
axes = np.asarray(axes).ravel()
for ax, z in zip(axes, z_values):
    mask = np.isclose(dataset.z, z)
    order = np.argsort(dataset.k_velocity[mask])
    kv = dataset.k_velocity[mask][order]
    ax.axhspan(-2, 2, color="0.7", alpha=0.15)
    ax.axhline(0, color="black", lw=1)
    ax.plot(kv, pulls["linear"][mask][order], "o-", ms=3.5, lw=1.2, label="linear")
    ax.plot(kv, pulls["one_loop"][mask][order], "s--", ms=3.2, lw=1.2, label="one-loop")
    ax.set_xscale("log")
    ax.set_xlabel(r"$k_v\,[\mathrm{s/km}]$")
    ax.set_ylabel(r"$(P_{\rm model}-P_{\rm data})/\sigma_{\rm total}$")
    ax.set_title(rf"$z={z:.1f}$")
for ax in axes[len(z_values):]:
    ax.set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.995),
    ncol=2, frameon=False,
)
fig.suptitle("Best-fit pulls", y=1.015)
fig.tight_layout(rect=[0, 0, 1, 0.955])
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR / "linear_vs_one_loop_pulls.png", dpi=180, bbox_inches="tight")
plt.show()


## 12. Względna różnica między krzywymi best fit

Krzywe zawierają już niezależnie dopasowane nuisance parameters. Nie jest to więc sama poprawka SPT, lecz końcowa zmiana przewidywanego $P_{1D}$.


In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.5 * nrows), sharey=True)
axes = np.asarray(axes).ravel()
for ax, z in zip(axes, z_values):
    mask = np.isclose(dataset.z, z)
    order = np.argsort(dataset.k_velocity[mask])
    kv = dataset.k_velocity[mask][order]
    relative = models["one_loop"][mask][order] / models["linear"][mask][order] - 1
    ax.axhline(0, color="black", lw=1)
    ax.plot(kv, 100 * relative, "o-", ms=4, color="tab:purple")
    ax.set_xscale("log")
    ax.set_xlabel(r"$k_v\,[\mathrm{s/km}]$")
    ax.set_ylabel(r"$100\,(P_{\rm 1L}/P_{\rm lin}-1)\,[\%]$")
    ax.set_title(rf"$z={z:.1f}$")
for ax in axes[len(z_values):]:
    ax.set_visible(False)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR / "best_fit_relative_difference.png", dpi=180, bbox_inches="tight")
plt.show()


## 13. Opcjonalny skan cutoffu

Po otrzymaniu teorii `production` lub `precision` ustaw `RUN_CUTOFF_SCAN=True`. Dla każdego cutoffu model one-loop jest ponownie fitowany; porównujemy minimum $\chi^2$, a nie parametry przy ustalonym punkcie. Dla pliku `smoke` wynik takiego skanu nie ma znaczenia naukowego.


In [ ]:
cutoff_rows = []
if RUN_CUTOFF_SCAN:
    for cutoff in CUTOFFS:
        cutoff_fitter = EffectiveModelFit(
            dataset, theory, mode="one_loop", k_uv_cut=cutoff,
            covariance_mode=COVARIANCE_MODE,
        )
        cutoff_runs = []
        for seed in CUTOFF_SCAN_SEEDS:
            result = cutoff_fitter.fit_once(
                seed=seed, bounds=cutoff_fitter.bounds_broad(),
                de_maxiter=DE_MAXITER, de_popsize=DE_POPSIZE,
            )
            cutoff_runs.append(result)
        best = min(cutoff_runs, key=lambda result: cutoff_fitter.chi2(result.x))
        cutoff_rows.append({
            "k_uv_cut": cutoff,
            "chi2_one_loop": cutoff_fitter.chi2(best.x),
            **{name: float(value) for name, value in zip(PARAMETER_NAMES, best.x)},
        })
    cutoff_table = pd.DataFrame(cutoff_rows)
    display(cutoff_table)
else:
    cutoff_table = pd.DataFrame()
    print("Cutoff scan skipped. Set RUN_CUTOFF_SCAN=True to run it.")


## 14. Zapis wyników

Zapisywane są zarówno podsumowania fitu, jak i krzywe w punktach danych. Po pierwszym wykonaniu można ustawić `RUN_NEW_FITS=False` i odtwarzać wykresy bez ponownej optymalizacji.


In [ ]:
result_table.to_csv(OUTPUT_DIR / "benchmark_comparison.csv", index=False)
summary_table.to_csv(OUTPUT_DIR / "fit_summary.csv", index=False)
multistart_table.to_csv(OUTPUT_DIR / "multistart_results.csv", index=False)
parameter_table.to_csv(OUTPUT_DIR / "best_fit_parameters.csv", index=False)
boundary_table.to_csv(OUTPUT_DIR / "boundary_diagnostics.csv", index=False)
chi2_by_z.to_csv(OUTPUT_DIR / "chi2_by_redshift.csv", index=False)
if not cutoff_table.empty:
    cutoff_table.to_csv(OUTPUT_DIR / "cutoff_scan.csv", index=False)

np.savez_compressed(
    OUTPUT_DIR / "best_fit_curves.npz",
    z=dataset.z, k_velocity=dataset.k_velocity, p1d_data=dataset.p1d,
    sigma_total=dataset.sigma_total, p1d_linear=models["linear"],
    p1d_one_loop=models["one_loop"], pull_linear=pulls["linear"],
    pull_one_loop=pulls["one_loop"], theta_linear=best_theta["linear"],
    theta_one_loop=best_theta["one_loop"], chi2_linear=chi2_linear,
    chi2_one_loop=chi2_loop, delta_chi2=delta_chi2,
)

comparison_record = {
    "theory_file": str(THEORY_PATH.resolve()),
    "theory_digest": theory.metadata["bundle_digest"],
    "model": model_name,
    "quality_hint": quality_hint,
    "k_uv_cut_hmpc": K_UV_CUT,
    "fits": {mode: summaries[mode] for mode in summaries},
    "delta_chi2_one_loop_minus_linear": delta_chi2,
    "benchmark_targets": targets,
    "multistart": multistart_table.to_dict(orient="records"),
    "chi2_by_redshift": chi2_by_z.to_dict(orient="records"),
}
write_fit_result(SAVED_COMPARISON_PATH, comparison_record)

print("Saved outputs:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)


## Co należy porównać po otrzymaniu `precision.npz`

Najpierw porównaj `production` i `precision` na poziomie surowych kanałów dla $k\leq2\,h/\mathrm{Mpc}$. Następnie uruchom ten notebook osobno dla obu plików i oceń:

1. zgodność $\chi^2_{\rm one-loop}$ i $\Delta\chi^2$;
2. stabilność dla $k_{\rm UV}=10,15,20\,h/\mathrm{Mpc}$;
3. brak systematycznego trendu w pullsach;
4. położenie parametrów względem granic;
5. zgodność z właściwym benchmarkiem kosmologicznym.
